In [1]:
# ============================================================
# CELL 1
# ============================================================

# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory
import sys
sys.path.append('/kaggle/input/entransformer-datasets')
import torch
from torch.utils.data import DataLoader


In [2]:
# ============================================================
# CELL 2
# ============================================================

%%capture
!pip install darts==0.39.0
!pip install gluonts
!pip install lightning


### Deterministic Seed Setting



In [3]:
# ============================================================
# CELL 3
# ============================================================

import os
import random
import numpy as np
import torch
import cv2
from transformers import set_seed
from datasets import disable_progress_bar
# VERY TOP OF NOTEBOOK, before importing torch/darts/lightning
import os
os.environ["PYTHONHASHSEED"] = "42"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
# optional for debugging only:
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

class Deterministic:
    def __init__(self):
        pass

    def init_all(self, seed=0, disable_list=['cuda_block']):
        random.seed(seed)
        os.environ['PYTHONHASHSEED'] = str(seed)
        os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
        if 'cuda_block' not in disable_list: # stuck when train deberta
            os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
        os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        if 'torch_deter_algo' not in disable_list: # consumn more gpu sometimes
            torch.use_deterministic_algorithms(True, warn_only=True)
        set_seed(seed)
        cv2.setRNGSeed(seed)
        disable_progress_bar()

deterministic = Deterministic()


### Importing Libraries

In [7]:
# ============================================================
# CELL 4
# ============================================================

import torch
import torch.nn as nn
import pandas as pd
from typing import Tuple, Optional
from darts import TimeSeries
from darts.models.forecasting.transformer_model import TransformerModel, _TransformerModule
from darts.metrics import mae, rmse, smape, mase, rmsse
from darts import TimeSeries
from darts.dataprocessing.transformers import Scaler


## Data Specific Params

In [5]:
# ============================================================
# CELL 5
# ============================================================

dataset_name = "electricity_nips"


In [6]:
# ============================================================
# CELL 6
# ============================================================

PRED_LEN = 24
LAGS = (1, 24, 168)


## Importing Data and Formatting

In [8]:
# ============================================================
# CELL 7
# ============================================================

import numpy as np
import pandas as pd

from gluonts.dataset.repository.datasets import get_dataset
from gluonts.dataset.multivariate_grouper import MultivariateGrouper

from darts import TimeSeries

def gluonts_item_to_darts_mv(item, freq: str) -> TimeSeries:
    # GluonTS start is often a pandas Period; convert safely
    start = item["start"].to_timestamp() if hasattr(item["start"], "to_timestamp") else pd.Timestamp(item["start"])

    # GluonTS multivariate target is typically shape (D, T)
    target = np.asarray(item["target"])
    if target.ndim != 2:
        raise ValueError(f"Expected multivariate target with ndim=2, got shape {target.shape}")

    values = target.T  # Darts expects (T, D)
    times = pd.date_range(start=start, periods=values.shape[0], freq=freq)
    cols = [f"dim_{i}" for i in range(values.shape[1])]

    return TimeSeries.from_times_and_values(times, values, columns=cols)

# Load dataset
ds = get_dataset(dataset_name, regenerate=False)
freq = ds.metadata.freq

# Determine number of rolling test windows (should be 7 for solar_nips)
train_list = list(ds.train)
test_list  = list(ds.test)
num_test_dates = len(test_list) // len(train_list)   # e.g., 959//137 = 7

# Group into multivariate series (like their code)
target_dim = int(ds.metadata.feat_static_cat[0].cardinality)  # 137 for solar
train_grouper = MultivariateGrouper(max_target_dim=target_dim)
test_grouper  = MultivariateGrouper(num_test_dates=num_test_dates, max_target_dim=target_dim)

train_mv_items = list(train_grouper(train_list))   # usually 1 item
test_mv_items  = list(test_grouper(test_list))     # usually num_test_dates items (7)

train_ts = gluonts_item_to_darts_mv(train_mv_items[0], freq)
test_ts_list = [gluonts_item_to_darts_mv(it, freq) for it in test_mv_items]  # len=7


In [11]:
# ============================================================
# CELL 8
# ============================================================

test_ts_list[2].shape


(4048, 370, 1)

In [9]:
# ============================================================
# CELL 9
# ============================================================

from darts.dataprocessing.transformers import Scaler

y_scaler = Scaler()                     # StandardScaler-like wrapper
train_y_sc = y_scaler.fit_transform(train_ts)

import numpy as np
from darts import TimeSeries, concatenate

def lag_covs_from_scaled_target(ts_sc: TimeSeries, lags=(1,24,168)) -> TimeSeries:
    shifted = []
    for L in lags:
        s = ts_sc.shift(L).with_columns_renamed(
            ts_sc.components, [f"{c}_lag{L}" for c in ts_sc.components]
        )
        shifted.append(s)

    common = shifted[0]
    for s in shifted[1:]:
        common = common.slice_intersect(s)
    shifted = [s.slice_intersect(common) for s in shifted]
    return concatenate(shifted, axis=1)

def fourier_from_index(idx) -> TimeSeries:
    hour = idx.hour.to_numpy()
    dow  = idx.dayofweek.to_numpy()
    X = np.vstack([
        np.sin(2*np.pi*hour/24.0),
        np.cos(2*np.pi*hour/24.0),
        np.sin(2*np.pi*dow/7.0),
        np.cos(2*np.pi*dow/7.0),
    ]).T
    return TimeSeries.from_times_and_values(idx, X, columns=["h_sin","h_cos","dow_sin","dow_cos"])

def dim_indicator_norm(idx, D: int) -> TimeSeries:
    v = (np.arange(D, dtype=np.float32) / (D-1)).astype(np.float32)  # 0..1
    X = np.tile(v, (len(idx), 1))
    cols = [f"dim_id_{i}" for i in range(D)]
    return TimeSeries.from_times_and_values(idx, X, columns=cols)

def build_past_covs_552(ts_sc: TimeSeries, lags=(1,24,168)) -> TimeSeries:
    lag_covs = lag_covs_from_scaled_target(ts_sc, lags)
    idx = lag_covs.time_index
    time_covs = fourier_from_index(idx)
    dim_covs  = dim_indicator_norm(idx, ts_sc.width)
    return concatenate([lag_covs, dim_covs, time_covs], axis=1)


def ts_upto(ts, end_time):
    # Newer Darts
    if hasattr(ts, "slice_end"):
        return ts.slice_end(end_time)
    # Some versions
    if hasattr(ts, "drop_after"):
        return ts.drop_after(end_time)
    if hasattr(ts, "split_after"):
        return ts.split_after(end_time)[0]
    # Works in basically all versions
    return ts.slice(ts.start_time(), end_time)


In [10]:
# ============================================================
# CELL 10
# ============================================================

import pandas as pd
from gluonts.dataset.repository.datasets import get_dataset

def gluon_to_wide_df(dataset):
    series_list = []
    
    for i, entry in enumerate(dataset):
        # 1. Create the time index using the start date and frequency
        idx = pd.date_range(
            start=entry["start"].to_timestamp(), 
            periods=len(entry["target"]), 
            freq=entry["start"].freqstr
        )
        
        # 2. Create a Series for each node, named by its index (0 to 136)
        series = pd.Series(entry["target"], index=idx, name=f"node_{i}")
        series_list.append(series)
    
    # 3. Concatenate all series into one DataFrame (T x N)
    return pd.concat(series_list, axis=1)

# Usage:
dataset = get_dataset(dataset_name, regenerate=False)
df_train = gluon_to_wide_df(dataset.train)

def get_7_test_windows(dataset, num_nodes=137):
    all_series = []
    
    # 1. Convert everything to a list of Series first
    for entry in dataset:
        idx = pd.date_range(
            start=entry["start"].to_timestamp(), 
            periods=len(entry["target"]), 
            freq=entry["start"].freqstr
        )
        # Give them a generic name for now
        all_series.append(pd.Series(entry["target"], index=idx))
    
    # 2. Split the list into 7 chunks of 137
    # This assumes the order is [Node0_W1, Node1_W1... Node136_W1, Node0_W2...]
    num_windows = len(all_series) // num_nodes
    windows = []
    
    for w in range(num_windows):
        start_idx = w * num_nodes
        end_idx = (w + 1) * num_nodes
        
        # Grab 137 series and concat them side-by-side
        window_df = pd.concat(all_series[start_idx:end_idx], axis=1)
        
        # Rename columns to node_0, node_1... node_136
        window_df.columns = [f"node_{i}" for i in range(num_nodes)]
        windows.append(window_df)
        
    return windows

# Execute
test_windows = get_7_test_windows(dataset.test, num_nodes=370)


In [11]:
# ============================================================
# CELL 11
# ============================================================

df_train


,node_0,node_1,node_2,node_3,node_4,node_5,node_6,node_7,node_8,node_9,...,node_360,node_361,node_362,node_363,node_364,node_365,node_366,node_367,node_368,node_369
2014-01-01 00:00:00,NaN,175.531921,31.993204,56.265984,47.902317,27.983105,209.358688,236.563446,6.930185,1161.392456,...,33.882782,7.200886,127.659576,90.614883,225.498398,NaN,1037.463989,90.314903,114.575340,47.912006
2014-01-01 01:00:00,NaN,164.893616,31.285391,55.200340,49.624294,27.719112,200.783005,236.570999,6.930185,638.185669,...,30.769230,6.462334,119.922630,87.702263,226.302246,NaN,1052.593628,93.010750,120.746574,46.234154
2014-01-01 02:00:00,NaN,154.787231,31.568516,55.413471,44.458359,27.719112,191.275162,246.397278,6.930185,573.839661,...,34.981686,9.231906,108.317215,84.466019,206.993576,NaN,946.685852,90.318741,126.232880,47.539150
2014-01-01 03:00:00,NaN,137.234039,29.728199,56.692242,39.292423,27.719112,180.462341,236.563446,6.673512,565.400818,...,30.402931,9.047267,102.030945,82.686081,208.609329,NaN,996.397705,86.470818,124.174660,53.504848
2014-01-01 04:00:00,NaN,138.297867,29.869762,55.200340,37.883533,27.719112,186.800888,232.787003,6.930185,558.016907,...,28.205128,8.493353,90.425529,86.084145,202.162384,NaN,1034.582153,76.866356,115.263702,52.199852
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2014-08-31 20:00:00,100.230415,122.340424,86.919594,58.184143,36.318096,38.542767,559.097717,667.273438,17.710472,4990.506348,...,293.223450,76.624817,146.518372,363.430420,706.535339,79.769737,2247.838623,173.329498,324.017120,127.143921
2014-08-31 21:00:00,95.046082,148.936172,83.097397,58.823528,45.554165,39.070751,525.913513,635.460754,17.710472,3943.038086,...,195.054947,78.286560,170.696320,361.974121,668.633423,79.358551,2292.507324,167.949310,318.178070,117.449661
2014-08-31 22:00:00,71.428574,128.723404,63.703285,60.102303,43.519100,39.334740,435.309479,511.072510,15.143737,3237.341797,...,133.699631,64.992615,168.278534,288.834961,625.104492,79.358551,2206.051758,139.512283,238.143829,90.231171
2014-08-31 23:00:00,52.995392,129.787231,45.583241,57.757885,45.241077,39.070751,413.870239,283.429016,13.347023,1869.198364,...,122.344322,60.930576,147.969055,246.440125,508.802246,72.779602,2267.291016,111.459297,137.886993,66.741241


np.False_

In [10]:
# ============================================================
# CELL 12
# ============================================================

df_train.shape


(5833, 370)

In [19]:
# ============================================================
# CELL 13
# ============================================================

test_ts_list[3]


,dim_0,dim_1,dim_2,dim_3,dim_4,...,dim_365,dim_366,dim_367,dim_368,dim_369
2014-03-19 09:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000
2014-03-19 10:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000
2014-03-19 11:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000
2014-03-19 12:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000
2014-03-19 13:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
2014-09-04 20:00:00,134.216583,148.936172,77.718010,99.104858,98.622414,...,74.424339,2963.977051,182.173584,322.301361,118.195374
2014-09-04 21:00:00,108.870964,167.553192,77.718010,96.547318,93.769569,...,78.536186,3011.527344,174.873276,311.308228,114.466812
2014-09-04 22:00:00,62.211983,147.340424,63.278595,86.743393,93.613022,...,73.190788,3283.861572,142.200455,225.010269,86.502609
2014-09-04 23:00:00,59.331799,135.638290,45.300114,65.643646,93.456482,...,69.078949,3203.890381,116.067589,142.684937,70.842651


In [20]:
# ============================================================
# CELL 14
# ============================================================

test_windows[3]


,node_0,node_1,node_2,node_3,node_4,node_5,node_6,node_7,node_8,node_9,...,node_360,node_361,node_362,node_363,node_364,node_365,node_366,node_367,node_368,node_369
2014-03-22 09:00:00,85.829491,130.851059,68.657982,86.743393,81.089546,29.567055,481.357208,543.557373,6.930185,1137.130859,...,77.838829,21.233383,106.866539,228.478958,600.112549,57.976974,1874.639771,156.036865,281.770538,82.401192
2014-03-22 10:00:00,110.599075,152.659576,66.251419,92.071609,83.594238,29.303062,475.018646,618.043823,6.930185,1502.109741,...,102.930405,43.205318,107.350098,230.906143,613.818298,64.967102,2043.227661,175.637482,297.222595,88.180466
2014-03-22 11:00:00,114.055298,163.297867,74.745186,91.858482,85.942390,29.303062,471.290070,602.129883,6.930185,1684.599121,...,159.706955,43.574593,103.481628,257.119751,587.202576,74.424339,2059.798340,187.553757,302.376709,88.926178
2014-03-22 12:00:00,99.078339,166.489365,74.886749,95.268539,89.699440,29.303062,477.255768,633.187317,6.930185,2412.447266,...,149.267395,54.098965,103.965187,252.588989,582.363342,83.881577,1972.622437,191.394012,321.270538,87.248322
2014-03-22 13:00:00,145.161285,151.063828,70.356735,92.284737,50.407013,29.831045,484.153625,606.684265,6.930185,2787.974609,...,149.450546,64.069427,92.359764,250.000000,565.442139,62.500000,1777.377563,199.470047,288.636993,88.739746
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2014-09-04 20:00:00,134.216583,148.936172,77.718010,99.104858,98.622414,38.278774,623.042480,689.229614,14.373716,4379.747070,...,281.501831,62.223042,140.232101,288.673126,721.848877,74.424339,2963.977051,182.173584,322.301361,118.195374
2014-09-04 21:00:00,108.870964,167.553192,77.718010,96.547318,93.769569,39.070751,570.842651,634.705444,13.090349,3412.447266,...,193.772888,63.884785,137.330750,280.097076,671.864929,78.536186,3011.527344,174.873276,311.308228,114.466812
2014-09-04 22:00:00,62.211983,147.340424,63.278595,86.743393,93.613022,36.694824,506.338562,522.416931,11.550308,3079.114014,...,134.432236,57.791729,130.077362,263.268616,616.229919,73.190788,3283.861572,142.200455,225.010269,86.502609
2014-09-04 23:00:00,59.331799,135.638290,45.300114,65.643646,93.456482,36.430836,429.530212,294.018127,11.550308,1774.261597,...,118.864471,48.005909,118.471954,224.919098,509.598083,69.078949,3203.890381,116.067589,142.684937,70.842651


In [11]:
# ============================================================
# CELL 15
# ============================================================

import numpy as np
import pandas as pd
from darts import TimeSeries

def impute_seasonal_mean(df, m: int) -> TimeSeries:
    """
    Imputes NaN values in a Darts TimeSeries of shape T x N using the seasonal mean.
    
    Parameters:
    - ts: The original Darts TimeSeries object.
    - m: The seasonal period (e.g., 12 for monthly data, 24 for hourly data).
    
    Returns:
    - A new TimeSeries with NaN values imputed, preserving original metadata.
    """
    
    # 2. Create a seasonal indicator array (0 to m-1) 
    # This works reliably because Darts enforces a strictly regular time index
    season_indicator = np.arange(len(df)) % m
    
    # 3. Group by the season, calculate the mean per column, and fill NaNs
    # transform('mean') broadcasts the group means back to the original index shape
    df_imputed = df.fillna(df.groupby(season_indicator).transform('mean'))
    
    # 4. Inject the values back into the TimeSeries to retain metadata
    return df_imputed


df_train = impute_seasonal_mean(df_train, m=24)


### Metric

In [12]:
# ============================================================
# CELL 16
# ============================================================

def crps(preds, targets, quantiles=(np.arange(20) / 20.0)[1:]):
    """
    preds: (B, N, T, D) or (B, N, T)
    targets: (B, T, D) or (B, T)
    """
    x = np.quantile(preds, quantiles, axis=1, method="nearest")  # -> (Q, B, T, D) or (Q, B, T)
    quantiles = np.expand_dims(quantiles, axis=list(range(1, len(preds.shape))))  # (Q,1,1,1)
    loss = 2 * np.sum(np.abs((x - targets) * ((targets <= x) - quantiles)), axis=2)  # sum over T
    return loss.mean() / np.abs(targets).sum(axis=1).mean()

def crps_sum(preds, targets, quantiles=(np.arange(20) / 20.0)[1:], frequency = 'D'):
    # preds: (B, N, T, D)
    # targets: (B, T, D)

    preds_sum = preds.sum(axis=-1)      # (B, N, T)
    targets_sum = targets.sum(axis=-1)  # (B, T)

    return crps(preds_sum, targets_sum, quantiles=quantiles)


def get_crps(model, test_windows, y_scaler, pred_len=24, lags=(1, 24, 168), num_samples=100, seed = 42, std = None):
    all_forecasts = []
    all_targets = []

    for i, window_df in enumerate(test_windows):
        full_ts = TimeSeries.from_dataframe(window_df).astype(np.float32)
        full_sc = y_scaler.transform(full_ts).astype(np.float32)
        full_pc = build_past_covs_552(full_sc, lags=lags).astype(np.float32)

        full_sc = full_sc.slice_intersect(full_pc)
        full_ts = full_ts.slice_intersect(full_sc)
        full_pc = full_pc.slice_intersect(full_sc)

        past_true_sc = full_sc[:-pred_len]
        gt_future    = full_ts[-pred_len:]
        
        forecast_start = gt_future.start_time()
        pc_past = ts_upto(full_pc, forecast_start)

        model.model.encoder[0].reset_seed(seed)
        if std is not None:
            model.model.encoder[0].reset_std(std)
        
        fc_sc = model.predict(
            n=pred_len,
            series=past_true_sc,
            past_covariates=pc_past,
            num_samples=num_samples,
            verbose = False,
            random_state = 1456445
        )

        fc = y_scaler.inverse_transform(fc_sc)
        fc = fc.with_values(np.clip(fc.all_values(), a_min=0, a_max=None))

        assert fc.time_index.equals(gt_future.time_index), \
            f"Forecast index mismatch in window {i}"

        all_forecasts.append(fc.all_values(copy=False))
        all_targets.append(gt_future.all_values(copy=False))

        # print(f"Processed window {i+1}/{len(test_windows)}")

    stacked_forecasts = np.stack(all_forecasts, axis=0)      # (B, T, D, N)
    preds_reshaped = np.transpose(stacked_forecasts, (0, 3, 1, 2))  # (B, N, T, D)

    stacked_targets = np.stack(all_targets, axis=0)          # (B, T, D, 1)
    targets_reshaped = np.squeeze(stacked_targets, axis=-1)  # (B, T, D)

    return crps_sum(preds_reshaped, targets_reshaped)


## Model

In [13]:
# ============================================================
# CELL 17
# ============================================================

def energy_score_loss(samples: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """
    samples: (M, batch, len, dim) - M stochastic samples
    target: (batch, len, dim) - Ground truth
    """
    M = samples.size(0)
    
    # Term 1: Mean distance to target E[||Y - y||]
    # Resulting shape: (batch, len)
    dist_to_target = torch.linalg.norm(samples - target.unsqueeze(0), dim=-1).mean(0)
    
    # Term 2: Mean pairwise distance between samples 0.5 * E[||Y - Y'||]
    # Flatten samples to (M, batch*len, dim) for efficient pairwise calculation
    s = samples.reshape(M, -1, samples.size(-1))
    # (M, 1, BL, D) - (1, M, BL, D) -> (M, M, BL, D)
    diff = s.unsqueeze(1) - s.unsqueeze(0)
    pairwise_dist = torch.linalg.norm(diff, dim=-1).mean(dim=(0, 1))
    dist_samples = pairwise_dist.view(target.size(0), target.size(1))
    
    # Final Energy Score (mean over batch and time)
    loss = dist_to_target - 0.5 * dist_samples
    return loss.mean()


import torch
import torch.nn as nn

class GaussianNoise(nn.Module):
    def __init__(self, std: float, seed: int | None = None):
        super().__init__()
        self.std = std
        self.seed = seed
        self._gen = None
        self._gen_device = None

    def _get_generator(self, device: torch.device):
        if self._gen is None or self._gen_device != device:
            self._gen = torch.Generator(device=device)
            self._gen_device = device
            if self.seed is not None:
                self._gen.manual_seed(self.seed)
        return self._gen

    def reset_seed(self, seed: int | None = None):
        if seed is not None:
            self.seed = seed
        if self.seed is None:
            raise ValueError("No seed set for this module.")
        if self._gen is not None:
            self._gen.manual_seed(self.seed)

    def reset_std(self, std: float | None = None):
        self.std = std

    def forward(self, x: torch.Tensor):
        g = self._get_generator(x.device)
        noise = torch.randn(
            x.shape,
            dtype=x.dtype,
            device=x.device,
            generator=g,
        )
        return x + noise * self.std


class UniformNoise(nn.Module):
    def __init__(self, std: float, seed: int | None = None):
        super().__init__()
        self.std = std
        self.seed = seed
        self._gen = None
        self._gen_device = None

    def _get_generator(self, device: torch.device):
        if self._gen is None or self._gen_device != device:
            self._gen = torch.Generator(device=device)
            self._gen_device = device
            if self.seed is not None:
                self._gen.manual_seed(self.seed)
        return self._gen

    def reset_seed(self, seed: int | None = None):
        if seed is not None:
            self.seed = seed
        if self.seed is None:
            raise ValueError("No seed set for this module.")
        if self._gen is not None:
            self._gen.manual_seed(self.seed)

    def reset_std(self, std: float | None = None):
        self.std = std

    def forward(self, x: torch.Tensor):
        g = self._get_generator(x.device)
        noise = torch.rand(
            x.shape,
            dtype=x.dtype,
            device=x.device,
            generator=g,
        )
        return x + (2 * self.std) * noise - self.std



class EnTransformerModule(_TransformerModule):
    def __init__(self, input_size, output_size, 
            nr_params, *args, num_samples_engression=10, noise_dist="gaussian", noise_std=0.1, **kwargs):
        super().__init__(input_size=input_size, 
            output_size=output_size, 
            nr_params=nr_params, *args, **kwargs)
        
        self.M = num_samples_engression
        self.noise_std = noise_std
        self.noise_dist = noise_dist
        
        # Inject noise at the input of the encoder (adds to input sequence)
        # self.encoder is usually a nn.Linear mapping input_dim to d_model
        if self.noise_dist == "gaussian":
            self.encoder = nn.Sequential(
                GaussianNoise(self.noise_std, 42),
                self.encoder
            )
        elif self.noise_dist == "uniform":
            self.encoder = nn.Sequential(
                UniformNoise(self.noise_std, 42),
                self.encoder
            )
        else:
            raise ValueError("noise_dist must be either `gaussian` or `uniform`.")
            

    def forward(self, x_in: tuple, *args, **kwargs):
        """Intercepts data during prediction to ensure float32."""
        x_in_float = tuple(
            t.float() if isinstance(t, torch.Tensor) and torch.is_floating_point(t) else t 
            for t in x_in
        )
        return super().forward(x_in_float, *args, **kwargs)

    def training_step(self, batch, batch_idx):
        # Unpack the 7-element batch
        past_target = batch[0]
        past_covs = batch[1]
        static_covs = batch[4]
        future_target = batch[-1]
        
        batch_size = future_target.size(0)

        # Normalize past and future targets
        past_target_norm = past_target 
        future_target_norm = future_target 

        # Expand inputs M times
        past_target_m = past_target_norm.repeat_interleave(self.M, dim=0)
        
        # Handle Covariates: Darts concatenates covariates to the target 
        # along the feature dimension (dim=2) before calling _create_transformer_inputs
        if past_covs is not None:
            past_covs_m = past_covs.repeat_interleave(self.M, dim=0)
            # Combine target and covariates into a single tensor
            data_m = torch.cat([past_target_m, past_covs_m], dim=2)
        else:
            data_m = past_target_m

        # Construct the input for the @io_processor decorated forward pass
        # It expects: (data_tensor, future_covariates, static_covariates)
        # We need to repeat these as well
        fut_covs_m = batch[3].repeat_interleave(self.M, dim=0) if batch[3] is not None else None
        static_covs_m = static_covs.repeat_interleave(self.M, dim=0) if static_covs is not None else None
        data_m = data_m.float()
        #fut_covs_m = fut_covs_m.float()
        #static_covs_m = static_covs_m.float()
        x_expanded = (data_m, fut_covs_m, static_covs_m)

        # Forward pass
        # Now 'data' is a Tensor, so data.permute(1, 0, 2) will work
        y_hat_raw = self(x_expanded) # (BM, T, D)
        
        # Reshape for Energy Score
        samples = y_hat_raw.view(batch_size, self.M, y_hat_raw.shape[1], y_hat_raw.shape[2]) # (B, M, T, D)
        samples = samples.permute(1, 0, 2, 3) # (M, B, T, D)
        
        
        loss = energy_score_loss(samples, future_target_norm)
        
        self.log("energy_score_train_loss", loss, prog_bar=True, on_epoch=True)
        return loss



# 4. Custom Darts Model Class
class EnTransformerModel(TransformerModel):
    def __init__(self, input_chunk_length, output_chunk_length, *args, num_samples_engression: int = 10, noise_dist='gaussian', noise_std: float = 0.1, random_state = 23, **kwargs):
        """
        Darts Transformer implementation with Engression.
        
        Parameters
        ----------
        num_samples_engression
            Number of samples (M) used to compute the Energy Score loss during training.
        noise_std
            Standard deviation of the Gaussian noise added to the input sequence.
        """
        
        self.num_samples_engression = num_samples_engression
        self.noise_std = noise_std
        self.noise_dist = noise_dist
        #self.noise_dist = noise_dist
        super().__init__(input_chunk_length=input_chunk_length,
                         output_chunk_length=output_chunk_length,
                         random_state = random_state,
                         pl_trainer_kwargs={"deterministic": True,
                                            "logger": False,
                                            "enable_progress_bar": False,
                                            "enable_model_summary": False,},
                         *args, **kwargs)
        # 1. Clean up model_params to avoid passing training-specific args to the PL Module
        # These are used by the Trainer/Model, not the neural network layers
        self.model_params.pop('num_samples_engression', None)
        self.model_params.pop('random_state', None)
        self.model_params.pop('noise_std', None)
        self.model_params.pop('noise_dist', None)
        self.model_params.pop('batch_size', None)
        self.model_params.pop('n_epochs', None)
        self.model_params.pop('optimizer_kwargs', None)
        self.model_params.pop('lr_scheduler_cls', None)
        self.model_params.pop('lr_scheduler_kwargs', None)


    def _create_model(self, train_sample):
        past_target = train_sample[0]
        past_covs   = train_sample[1]

        # print(f"past target shape {past_target.shape}")
        # print(f"past cov shape {past_covs.shape}")
    
        # if past_target is (B, T, D):
        target_dim = past_target.shape[-1]
        past_cov_dim = past_covs.shape[-1] if past_covs is not None else 0
        input_size = target_dim + past_cov_dim
    
        future_target = train_sample[-1]  # works for both 6 and 7 if future_target is last in train_sample
        output_size = future_target.shape[-1]
    
        return EnTransformerModule(
            input_size=input_size,
            output_size=output_size,
            nr_params=getattr(self, "nr_params", 1),
            num_samples_engression=self.num_samples_engression,
            noise_std=self.noise_std,
            noise_dist=self.noise_dist,
            **self.model_params
        )


In [14]:
# ============================================================
# CELL 18
# ============================================================

# train_series = TimeSeries.from_dataframe(df_train)
train_pc = build_past_covs_552(train_y_sc)
train_y_sc = train_y_sc.slice_intersect(train_pc)
train_pc   = train_pc.slice_intersect(train_y_sc)


In [11]:
# ============================================================
# CELL 11 — TRAIN / CALIBRATION SPLIT
# ============================================================

cal_end = train_y_sc.end_time()

cal_start = (
    cal_end
    - pd.Timedelta(hours=(CAL_LENGTH * PRED_LEN) - 1)
)

# Calibration target
calibration_y_sc = train_y_sc[
    cal_start:
]

# Actual model-training target:
# everything BEFORE calibration region
base_train_y_sc = train_y_sc[
    :cal_start - pd.Timedelta(hours=1)
]

# Corresponding covariates
base_train_pc = train_pc[
    :base_train_y_sc.end_time()
]

calibration_pc = train_pc[
    cal_start:
]

print(
    "Base training:",
    base_train_y_sc.start_time(),
    "->",
    base_train_y_sc.end_time()
)

print(
    "Calibration:",
    calibration_y_sc.start_time(),
    "->",
    calibration_y_sc.end_time()
)

print("Base train shape:", base_train_y_sc.shape)
print("Calibration shape:", calibration_y_sc.shape)


Base training: 2006-01-08 00:00:00 -> 2006-10-13 00:00:00
Calibration: 2006-10-13 01:00:00 -> 2006-10-20 00:00:00
Base train shape: (6673, 137, 1)
Calibration shape: (168, 137, 1)


In [16]:
# ============================================================
# CELL 13 — METRIC FUNCTIONS
# ============================================================

def _sync_tensors(*args):

    tensors = [
        torch.as_tensor(x)
        if not isinstance(x, torch.Tensor)
        else x
        for x in args
    ]

    device = tensors[0].device

    return [
        t.to(
            device,
            dtype=torch.float32
        )
        for t in tensors
    ]


def get_point_forecast(
    y_pred,
    method="median"
):

    if method == "mean":
        return torch.mean(
            y_pred,
            dim=-1
        )

    if method == "median":
        return torch.median(
            y_pred,
            dim=-1
        ).values

    if isinstance(method, float):
        return torch.quantile(
            y_pred,
            method,
            dim=-1
        )

    raise ValueError(
        "Invalid point method"
    )


def aggregate(x):
    return torch.mean(x)


def metric_mae(
    y_true,
    y_pred
):

    y_true, y_pred = _sync_tensors(
        y_true,
        y_pred
    )

    yp = get_point_forecast(
        y_pred,
        POINT_METHOD
    )

    return aggregate(
        torch.abs(
            y_true - yp
        )
    ).item()


def metric_mse(
    y_true,
    y_pred
):

    y_true, y_pred = _sync_tensors(
        y_true,
        y_pred
    )

    yp = get_point_forecast(
        y_pred,
        POINT_METHOD
    )

    return aggregate(
        (y_true - yp) ** 2
    ).item()


def metric_rmse(
    y_true,
    y_pred
):

    return np.sqrt(
        metric_mse(
            y_true,
            y_pred
        )
    )


def metric_mape(
    y_true,
    y_pred,
    eps=1e-8
):

    y_true, y_pred = _sync_tensors(
        y_true,
        y_pred
    )

    yp = get_point_forecast(
        y_pred,
        POINT_METHOD
    )

    value = torch.abs(
        (y_true - yp)
        / torch.clamp(
            torch.abs(y_true),
            min=eps
        )
    )

    return (
        torch.mean(value) * 100
    ).item()


def metric_smape(
    y_true,
    y_pred,
    eps=1e-8
):

    y_true, y_pred = _sync_tensors(
        y_true,
        y_pred
    )

    yp = get_point_forecast(
        y_pred,
        POINT_METHOD
    )

    numerator = torch.abs(
        yp - y_true
    )

    denominator = torch.clamp(
        (
            torch.abs(y_true)
            + torch.abs(yp)
        ) / 2,
        min=eps
    )

    return (
        torch.mean(
            numerator / denominator
        ) * 100
    ).item()


def metric_mase(
    y_true,
    y_pred,
    y_train,
    eps=1e-8
):

    y_true, y_pred, y_train = _sync_tensors(
        y_true,
        y_pred,
        y_train
    )

    yp = get_point_forecast(
        y_pred,
        POINT_METHOD
    )

    mae_forecast = torch.abs(
        y_true - yp
    )

    diff = torch.abs(
        y_train[1:]
        - y_train[:-1]
    )

    scale = torch.mean(
        diff,
        dim=0
    )

    scale = torch.clamp(
        scale,
        min=eps
    )

    value = (
        mae_forecast
        / scale.unsqueeze(0)
    )

    return torch.mean(
        value
    ).item()


def metric_rmsse(
    y_true,
    y_pred,
    y_train,
    eps=1e-8
):

    y_true, y_pred, y_train = _sync_tensors(
        y_true,
        y_pred,
        y_train
    )

    yp = get_point_forecast(
        y_pred,
        POINT_METHOD
    )

    mse_forecast = (
        y_true - yp
    ) ** 2

    diff = (
        y_train[1:]
        - y_train[:-1]
    ) ** 2

    scale = torch.mean(
        diff,
        dim=0
    )

    scale = torch.clamp(
        scale,
        min=eps
    )

    value = torch.sqrt(
        mse_forecast
        / scale.unsqueeze(0)
    )

    return torch.mean(
        value
    ).item()


def metric_crps(
    y_true,
    y_pred
):

    y_true, y_pred = _sync_tensors(
        y_true,
        y_pred
    )

    y_true_ext = (
        y_true.unsqueeze(-1)
    )

    abs_diff_true = torch.mean(
        torch.abs(
            y_pred
            - y_true_ext
        ),
        dim=-1
    )

    y_i = y_pred.unsqueeze(-1)
    y_j = y_pred.unsqueeze(-2)

    abs_diff_samples = torch.mean(
        torch.abs(
            y_i - y_j
        ),
        dim=(-1, -2)
    )

    score = (
        abs_diff_true
        - 0.5 * abs_diff_samples
    )

    return torch.mean(score).item()


def metric_picp(
    y_true,
    y_pred,
    alpha=0.05
):

    y_true, y_pred = _sync_tensors(
        y_true,
        y_pred
    )

    lower = torch.quantile(
        y_pred,
        alpha / 2,
        dim=-1
    )

    upper = torch.quantile(
        y_pred,
        1 - alpha / 2,
        dim=-1
    )

    inside = (
        (y_true >= lower)
        & (y_true <= upper)
    ).float()

    return torch.mean(
        inside
    ).item()


def metric_mpiw(
    y_pred,
    alpha=0.05
):

    y_pred = _sync_tensors(
        y_pred
    )[0]

    lower = torch.quantile(
        y_pred,
        alpha / 2,
        dim=-1
    )

    upper = torch.quantile(
        y_pred,
        1 - alpha / 2,
        dim=-1
    )

    return torch.mean(
        upper - lower
    ).item()


def metric_mis(
    y_true,
    y_pred,
    alpha=0.05
):

    y_true, y_pred = _sync_tensors(
        y_true,
        y_pred
    )

    lower = torch.quantile(
        y_pred,
        alpha / 2,
        dim=-1
    )

    upper = torch.quantile(
        y_pred,
        1 - alpha / 2,
        dim=-1
    )

    widths = upper - lower

    below = (
        y_true < lower
    ).float()

    above = (
        y_true > upper
    ).float()

    penalty_below = (
        (2 / alpha)
        * (lower - y_true)
        * below
    )

    penalty_above = (
        (2 / alpha)
        * (y_true - upper)
        * above
    )

    return torch.mean(
        widths
        + penalty_below
        + penalty_above
    ).item()


def metric_rho_risk(
    y_true,
    y_pred,
    quantiles=(0.1, 0.5, 0.9),
    eps=1e-8
):

    y_true, y_pred = _sync_tensors(
        y_true,
        y_pred
    )

    total_true = torch.clamp(
        torch.sum(
            torch.abs(y_true)
        ),
        min=eps
    )

    values = []

    for q in quantiles:

        yq = torch.quantile(
            y_pred,
            q,
            dim=-1
        )

        error = (
            y_true - yq
        )

        loss = torch.maximum(
            q * error,
            (q - 1) * error
        )

        values.append(
            2 * torch.sum(loss)
            / total_true
        )

    return torch.mean(
        torch.stack(values)
    ).item()


In [17]:
# ============================================================
# CELL 15 — MODEL FACTORY
# ============================================================

def make_model(
    model_name,
    likelihood=None
):

    common = dict(
        input_chunk_length=24,
        output_chunk_length=PRED_LEN,
        n_epochs=N_EPOCHS,
        batch_size=BATCH_SIZE,
        optimizer_kwargs={
            "lr": LEARNING_RATE
        },
        random_state=SEED,
        force_reset=True,
    )

    if likelihood is not None:
        common["likelihood"] = likelihood

    if model_name == "NBEATS":
        return NBEATSModel(
            **common
        )

    elif model_name == "NHiTS":
        return NHiTSModel(
            **common
        )

    elif model_name == "TCN":
        return TCNModel(
            input_chunk_length=30,
            output_chunk_length=PRED_LEN,
            n_epochs=N_EPOCHS,
            batch_size=BATCH_SIZE,
            optimizer_kwargs={
                "lr": LEARNING_RATE
            },
            random_state=SEED,
            likelihood=likelihood,
        )

    elif model_name == "Transformer":
        return TransformerModel(
            **common
        )

    elif model_name == "RNN":
        return RNNModel(
            model="LSTM",
            **common
        )

    elif model_name == "BlockRNN":
        return BlockRNNModel(
            model="LSTM",
            **common
        )

    elif model_name == "DLinear":
        return DLinearModel(
            **common
        )

    elif model_name == "NLinear":
        return NLinearModel(
            normalize=False,
            **common
        )

    elif model_name == "TiDE":
        return TiDEModel(
            **common
        )

    elif model_name == "TSMixer":
        return TSMixerModel(
            **common
        )

    elif model_name == "TFT":

        return TFTModel(
            input_chunk_length=24,
            output_chunk_length=PRED_LEN,
            hidden_size=32,
            lstm_layers=1,
            num_attention_heads=4,
            dropout=0.1,
            batch_size=BATCH_SIZE,
            n_epochs=N_EPOCHS,
            optimizer_kwargs={
                "lr": LEARNING_RATE
            },
            random_state=SEED,
            likelihood=likelihood,
            add_relative_index=True,
        )

    elif model_name == "Chronos2":
        return Chronos2Model(
            input_chunk_length=24,
            output_chunk_length=PRED_LEN,
            likelihood=likelihood,
            random_state=SEED,
            n_epochs=N_EPOCHS,
        )

    elif model_name == "TimesFM2p5":
        return TimesFM2p5Model(
            input_chunk_length=24,
            output_chunk_length=PRED_LEN,
            likelihood=likelihood,
            random_state=SEED,
            n_epochs=N_EPOCHS,
        )


    elif model_name == "PatchTSTFM":
        import importlib.util
        import sys
        spec = importlib.util.spec_from_file_location("patchtst", "../../models/darts-original/patchtst_fm_model.py")
        patchtst = importlib.util.module_from_spec(spec)
        sys.modules["patchtst"] = patchtst
        spec.loader.exec_module(patchtst)
        return patchtst.PatchTSTFMModel(
            input_chunk_length=24,
            output_chunk_length=PRED_LEN,
            likelihood=likelihood,
            n_epochs=N_EPOCHS,
        )

    elif model_name == "TiREx":
        import importlib.util
        import sys
        spec = importlib.util.spec_from_file_location("tirex", "../../models/darts-original/tirex_model.py")
        tirex = importlib.util.module_from_spec(spec)
        sys.modules["tirex"] = tirex
        spec.loader.exec_module(tirex)
        return tirex.TiRExModel(
            input_chunk_length=24,
            output_chunk_length=PRED_LEN,
            likelihood=likelihood,
            n_epochs=N_EPOCHS,
            accept_license=True,
        )

    else:
        raise ValueError(
            f"Unknown model: {model_name}"
        )


In [18]:
# ============================================================
# CELL 16 — MODELS
# ============================================================

MODELS = [
    "BlockRNN",
    "BlockLSTM",
    "BlockGRU",
    "DLinear",
    "NBEATS",
    "NHiTS",
    "NLinear",
    "RNN",
    "LSTM",
    "GRU",
    "TCN",
    "TFT",
    "TiDE",
    "TSMixer",
    "Transformer",
]



print("Models:")
for x in MODELS:
    print(" -", x)


Models:
 - BlockRNN
 - BlockLSTM
 - BlockGRU
 - DLinear
 - NBEATS
 - NHiTS
 - NLinear
 - RNN
 - LSTM
 - GRU
 - TCN
 - TFT
 - TiDE
 - TSMixer
 - Transformer


In [19]:
# ============================================================
# CELL 17 - COVARIATE SUPPORT
# ============================================================

def get_fit_kwargs(
    model,
    target,
    past_covs=None
):

    kwargs = {
        "series": target,
        "verbose": True,
        "dataloader_kwargs": {"num_workers": 0},
    }

    if getattr(
        model,
        "supports_past_covariates",
        False
    ):
        if past_covs is not None:
            kwargs["past_covariates"] = past_covs
        else:
            kwargs["past_covariates"] = base_train_pc

    return kwargs


In [45]:
# ============================================================
# CELL 18 — TRAIN / EVALUATE LIKELIHOOD MODEL
# ============================================================

def evaluate_base_model(
    model,
    test_windows,
    likelihood_name,
    model_name,
    seeds=list(range(40, 51))
):

    all_seed_summaries = []

    for seed in seeds:
        window_metrics = []
        total_inference_time = 0.0

        for window_id, window_df in enumerate(
            test_windows
        ):

            full_ts = (
                TimeSeries
                .from_dataframe(window_df)
                .astype(np.float32)
            )

            full_sc = (
                y_scaler
                .transform(full_ts)
                .astype(np.float32)
            )

            full_pc = (
                build_past_covs(
                    full_sc,
                    lags=LAGS
                )
                .astype(np.float32)
            )

            full_sc = (
                full_sc
                .slice_intersect(full_pc)
            )

            full_ts = (
                full_ts
                .slice_intersect(full_sc)
            )

            full_pc = (
                full_pc
                .slice_intersect(full_sc)
            )

            past_sc = full_sc[:-PRED_LEN]

            past_original = (
                full_ts[:-PRED_LEN]
            )

            gt_future = (
                full_ts[-PRED_LEN:]
            )

            forecast_start = (
                gt_future.start_time()
            )

            pc_past = (
                full_pc
                .drop_after(forecast_start, keep_point=False)
            )

            predict_kwargs = {}

            if getattr(
                model,
                "supports_past_covariates",
                False
            ):
                predict_kwargs[
                    "past_covariates"
                ] = pc_past

            start = time.time()

            fc_sc = model.predict(
                n=PRED_LEN,
                series=past_sc,
                num_samples=NUM_PRED_SAMPLES,
                random_state=seed,
                verbose=False,
                **predict_kwargs,
            )

            inference_time = (
                time.time() - start
            )

            total_inference_time += (
                inference_time
            )

            fc = y_scaler.inverse_transform(
                fc_sc
            )

            fc = fc.with_values(np.clip(fc.all_values(), a_min=0, a_max=None))

            y_pred = fc.all_values(
                copy=False
            )

            y_true = (
                gt_future
                .all_values(copy=False)
                .squeeze(-1)
            )

            y_train = (
                past_original
                .all_values(copy=False)
                .squeeze(-1)
            )

            metrics = {

                "MAE": metric_mae(y_true, y_pred),
                "MSE": metric_mse(y_true, y_pred),
                "RMSE": metric_rmse(y_true, y_pred),
                "MAPE": metric_mape(y_true, y_pred),
                "sMAPE": metric_smape(y_true, y_pred),
                "MASE": metric_mase(y_true, y_pred, y_train),
                "RMSSE": metric_rmsse(y_true, y_pred, y_train),
                "CRPS": metric_crps(y_true, y_pred),
                "PICP": metric_picp(y_true, y_pred, alpha=ALPHA),
                "MIS": metric_mis(y_true, y_pred, alpha=ALPHA),
                "MPIW": metric_mpiw(y_pred, alpha=ALPHA),
                "Rho_Risk": metric_rho_risk(y_true, y_pred),
            }

            window_metrics.append(
                metrics
            )

        summary = pd.DataFrame(
            window_metrics
        ).mean()

        summary[
            "Inference Time (s)"
        ] = (
            total_inference_time
            / len(test_windows)
        )
        
        all_seed_summaries.append(summary)

    concat_df = pd.concat(all_seed_summaries, axis=1)
    mean_series = concat_df.mean(axis=1)
    if len(seeds) > 1:
        std_series = concat_df.std(axis=1)
        formatted_series = mean_series.combine(std_series, lambda m, s: f"${m:.3f}\\pm{s:.3f}$")
    else:
        formatted_series = mean_series.apply(lambda m: f"${m:.3f}\\pm0.000$")

    return formatted_series



In [46]:
# ============================================================
# CELL 25
# ============================================================

from darts.utils.likelihood_models import (
    GaussianLikelihood,
    QuantileRegression
)

LIKELIHOODS = {
    "Gaussian": GaussianLikelihood(),

    "Quantile": QuantileRegression(
        quantiles=QUANTILES
    ),

}


In [ ]:
# ============================================================
# CELL 19 — RUN ALL LIKELIHOOD BASELINES
# ============================================================
# ============================================================
# SUPPRESS WARNINGS / LIGHTNING LOGS
# ============================================================

import warnings
import logging
import os

# Python warnings
warnings.filterwarnings("ignore")

# PyTorch Lightning logs
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)

# Suppress Lightning distributed/GPU informational messages
os.environ["PYTHONWARNINGS"] = "ignore"

results = []

CSV_FILE = (
    "Electricity_Darts_Likelihood_Baselines.csv"
)

for model_name in MODELS:

    for likelihood_name in LIKELIHOODS:
        # Foundation models only support Quantile in this setup
        if model_name in ["Chronos2", "TimesFM2p5", "PatchTSTFM", "TiREx"] and likelihood_name != "Quantile":
            print(f"Skipping {likelihood_name} for {model_name} (Zero-shot Foundation Models default to Quantile)")
            continue


        print("\n")
        print("=" * 70)
        print(
            f"MODEL: {model_name} | "
            f"LIKELIHOOD: {likelihood_name}"
        )
        print("=" * 70)

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        try:

            likelihood = (
                LIKELIHOODS[
                    likelihood_name
                ]
            )

            model = make_model(
                model_name,
                likelihood=likelihood
            )

            print("Training...")

            start_train = time.time()

            fit_kwargs = (
                get_fit_kwargs(
                    model,
                    train_y_sc,
                    past_covs=train_pc
                )
            )

            model.fit(
                **fit_kwargs
            )

            training_time = (
                time.time()
                - start_train
            )

            print(
                f"Training time: "
                f"{training_time:.2f}s"
            )

            print("Evaluating...")

            metrics = (
                evaluate_base_model(
                    model=model,
                    test_windows=test_windows,
                    likelihood_name=likelihood_name,
                    model_name=model_name,
                )
            )

            row = {
                "Model": model_name,
                "Method": likelihood_name,
                "N_Epochs": N_EPOCHS,
                "Batch Size": BATCH_SIZE,
                "Prediction Samples":
                    NUM_PRED_SAMPLES,
                "Training Time (s)":
                    training_time,
            }

            for metric_name, value in (
                metrics.items()
            ):
                row[metric_name] = value

            results.append(row)

            pd.DataFrame(
                results
            ).to_csv(
                CSV_FILE,
                index=False
            )

            print(
                pd.DataFrame(
                    [row]
                ).T
            )

        except Exception as e:

            print(
                f"FAILED: "
                f"{model_name} / "
                f"{likelihood_name}"
            )

            print(
                type(e).__name__,
                str(e)
            )

            continue

print("\nFinished likelihood benchmark.")

likelihood_results = pd.DataFrame(
    results
)

likelihood_results




MODEL: BlockRNN | LIKELIHOOD: Gaussian
Training...


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Training time: 29.31s
Evaluating...
                                                  0
Model                                      BlockRNN
Method                                     Gaussian
N_Epochs                                         30
Batch Size                                       64
Prediction Samples                              100
Training Time (s)                         29.311468
MAE                                $14.973\pm0.014$
MSE                               $934.159\pm2.872$
RMSE                               $30.198\pm0.043$
MAPE                $2517249589.195\pm22627134.040$
sMAPE                              $89.042\pm0.536$
MASE                                $1.175\pm0.001$
RMSSE                               $0.695\pm0.001$
CRPS                               $10.223\pm0.011$
PICP                                $0.936\pm0.001$
MIS                                $76.573\pm0.506$
MPIW                               $56.867\pm0.140$
Rho_Risk                    

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

...

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Training time: 433.61s
Evaluating...
                                                      0
Model                                           DLinear
Method                                         Quantile
N_Epochs                                             30
Batch Size                                           64
Prediction Samples                                  100
Training Time (s)                            433.613694
MAE                                    $41.194\pm0.315$
MSE                                $4492.740\pm103.917$
RMSE                                   $66.778\pm0.767$
MAPE                $166827123858.286\pm2460620710.164$
sMAPE                                 $103.004\pm0.498$
MASE                                    $3.261\pm0.022$
RMSSE                                   $1.929\pm0.013$
CRPS                                   $36.812\pm0.136$
PICP                                    $0.892\pm0.000$
MIS                                   $387.357\pm0.191$
MPIW       

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

In [ ]:
# ============================================================
# CELL 20 — CONFORMAL CALIBRATION HELPER
# ============================================================

def calibrate_conformal_model(
    base_model,
    conformal_type="naive"
):

    if conformal_type == "naive":

        cp_model = ConformalNaiveModel(
            model=base_model,
            quantiles=QUANTILES,
            symmetric=True,
            cal_length=CAL_LENGTH,
            cal_stride=PRED_LEN,
            cal_num_samples=NUM_PRED_SAMPLES,
            random_state=SEED,
        )

    elif conformal_type == "qr":

        cp_model = ConformalQRModel(
            model=base_model,
            quantiles=QUANTILES,
            symmetric=True,
            cal_length=CAL_LENGTH,
            cal_stride=PRED_LEN,
            cal_num_samples=NUM_PRED_SAMPLES,
            random_state=SEED,
        )

    else:

        raise ValueError(
            "Unknown conformal type"
        )

    return cp_model


In [ ]:
# ============================================================
# CELL 21 — CONFORMAL NAIVE BASE MODEL FACTORY
# ============================================================

def make_conformal_naive_base(
    model_name
):

    # Gaussian base model
    likelihood = GaussianLikelihood()

    return make_model(
        model_name,
        likelihood=likelihood
    )


In [ ]:
# ============================================================
# CELL 22 — EVALUATE CONFORMAL MODEL
# ============================================================

def evaluate_conformal_model(
    cp_model,
    test_windows,
    conformal_name,
    seeds=list(range(40, 51))
):

    all_seed_summaries = []

    for seed in seeds:
        window_metrics = []
        total_inference_time = 0.0

        for window_id, window_df in enumerate(
            test_windows
        ):

            full_ts = (
                TimeSeries
                .from_dataframe(window_df)
                .astype(np.float32)
            )

            full_sc = (
                y_scaler
                .transform(full_ts)
                .astype(np.float32)
            )

            full_pc = (
                build_past_covs(
                    full_sc,
                    lags=LAGS
                )
                .astype(np.float32)
            )

            full_sc = (
                full_sc
                .slice_intersect(full_pc)
            )

            full_ts = (
                full_ts
                .slice_intersect(full_sc)
            )

            full_pc = (
                full_pc
                .slice_intersect(full_sc)
            )

            past_sc = full_sc[:-PRED_LEN]

            past_original = (
                full_ts[:-PRED_LEN]
            )

            gt_future = (
                full_ts[-PRED_LEN:]
            )

            forecast_start = (
                gt_future.start_time()
            )

            pc_past = (
                full_pc
                .drop_after(forecast_start, keep_point=False)
            )

            predict_kwargs = {}

            if getattr(
                cp_model,
                "supports_past_covariates",
                False
            ):
                predict_kwargs[
                    "past_covariates"
                ] = pc_past

            start = time.time()

            fc_sc = cp_model.predict(
                n=PRED_LEN,
                series=past_sc,
                num_samples=NUM_PRED_SAMPLES,
                random_state=seed,
                verbose=False,
                **predict_kwargs,
            )

            inference_time = (
                time.time() - start
            )

            total_inference_time += (
                inference_time
            )

            fc = y_scaler.inverse_transform(
                fc_sc
            )

            fc = fc.with_values(np.clip(fc.all_values(), a_min=0, a_max=None))

            y_pred = fc.all_values(
                copy=False
            )

            y_true = (
                gt_future
                .all_values(copy=False)
                .squeeze(-1)
            )

            y_train = (
                past_original
                .all_values(copy=False)
                .squeeze(-1)
            )

            metrics = {

                "MAE": metric_mae(y_true, y_pred),
                "MSE": metric_mse(y_true, y_pred),
                "RMSE": metric_rmse(y_true, y_pred),
                "MAPE": metric_mape(y_true, y_pred),
                "sMAPE": metric_smape(y_true, y_pred),
                "MASE": metric_mase(y_true, y_pred, y_train),
                "RMSSE": metric_rmsse(y_true, y_pred, y_train),
                "CRPS": metric_crps(y_true, y_pred),
                "PICP": metric_picp(y_true, y_pred, alpha=ALPHA),
                "MIS": metric_mis(y_true, y_pred, alpha=ALPHA),
                "MPIW": metric_mpiw(y_pred, alpha=ALPHA),
                "Rho_Risk": metric_rho_risk(y_true, y_pred),
            }

            window_metrics.append(
                metrics
            )

        summary = pd.DataFrame(
            window_metrics
        ).mean()

        summary[
            "Inference Time (s)"
        ] = (
            total_inference_time
            / len(test_windows)
        )
        
        all_seed_summaries.append(summary)

    concat_df = pd.concat(all_seed_summaries, axis=1)
    mean_series = concat_df.mean(axis=1)
    if len(seeds) > 1:
        std_series = concat_df.std(axis=1)
        formatted_series = mean_series.combine(std_series, lambda m, s: f"${m:.3f}\\pm{s:.3f}$")
    else:
        formatted_series = mean_series.apply(lambda m: f"${m:.3f}\\pm0.000$")

    return formatted_series



In [ ]:
# ============================================================
# CELL 23 — CONFORMAL NAIVE
# ============================================================

conformal_results = []

CONFORMAL_CSV = (
    "Solar_Darts_Conformal_Baselines.csv"
)

for model_name in MODELS:

    print("\n")
    print("=" * 70)
    print(
        f"CONFORMAL NAIVE | {model_name}"
    )
    print("=" * 70)

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    try:

        # ----------------------------------------------------
        # 1. Train base model ONLY on base training region
        # ----------------------------------------------------

        base_model = (
            make_conformal_naive_base(
                model_name
            )
        )

        print(
            "Training base model..."
        )

        start_train = time.time()

        fit_kwargs = (
            get_fit_kwargs(
                base_model,
                base_train_y_sc
            )
        )

        base_model.fit(
            **fit_kwargs
        )

        training_time = (
            time.time()
            - start_train
        )

        print(
            f"Base training time: "
            f"{training_time:.2f}s"
        )

        # ----------------------------------------------------
        # 2. Create conformal wrapper
        # ----------------------------------------------------

        cp_model = (
            calibrate_conformal_model(
                base_model,
                conformal_type="naive"
            )
        )

        # ----------------------------------------------------
        # 3. Calibrate using HELD-OUT calibration region
        # ----------------------------------------------------

        print(
            "Calibrating..."
        )

        calibration_kwargs = {}

        if getattr(
            cp_model,
            "supports_past_covariates",
            False
        ):
            calibration_kwargs[
                "past_covariates"
            ] = calibration_pc

        # Darts conformal model uses the
        # calibration series supplied to predict().
        #
        # We perform one calibration pass by asking
        # it to generate forecasts over the calibration
        # region.


In [ ]:
# ============================================================
# CELL 24 — CHECK DARTS VERSION
# ============================================================

import darts

print(
    "Darts version:",
    darts.__version__
)

print(
    "ConformalNaiveModel:",
    ConformalNaiveModel
)

print(
    "ConformalQRModel:",
    ConformalQRModel
)


In [ ]:
# ============================================================
# CELL 25 — CONFORMAL QR BASE MODEL
# ============================================================

def make_conformal_qr_base(
    model_name
):

    likelihood = QuantileRegression(
        quantiles=QUANTILES
    )

    return make_model(
        model_name,
        likelihood=likelihood
    )


In [ ]:
# ============================================================
# CELL 26 — CONFORMAL QR
# ============================================================

for model_name in MODELS:

    print("\n")
    print("=" * 70)
    print(
        f"CONFORMAL QR | {model_name}"
    )
    print("=" * 70)

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    try:

        # ----------------------------------------------------
        # 1. Train probabilistic quantile base model
        # ----------------------------------------------------

        base_model = (
            make_conformal_qr_base(
                model_name
            )
        )

        print(
            "Training quantile base model..."
        )

        start_train = time.time()

        fit_kwargs = (
            get_fit_kwargs(
                base_model,
                base_train_y_sc
            )
        )

        base_model.fit(
            **fit_kwargs
        )

        training_time = (
            time.time()
            - start_train
        )

        print(
            f"Training time: "
            f"{training_time:.2f}s"
        )

        # ----------------------------------------------------
        # 2. Conformal QR wrapper
        # ----------------------------------------------------

        cp_model = (
            ConformalQRModel(
                model=base_model,
                quantiles=QUANTILES,
                symmetric=True,
                cal_length=CAL_LENGTH,
                cal_stride=PRED_LEN,
                cal_num_samples=NUM_PRED_SAMPLES,
                random_state=SEED,
            )
        )

        # ----------------------------------------------------
        # 3. Calibration
        # ----------------------------------------------------

        print(
            "Calibrating conformal QR..."
        )

        calibration_kwargs = {}

        if getattr(
            cp_model,
            "supports_past_covariates",
            False
        ):
            calibration_kwargs[
                "past_covariates"
            ] = calibration_pc


In [ ]:
# ============================================================
# CELL 27 — COMBINE RESULTS
# ============================================================

all_baseline_results = pd.concat(
    [
        likelihood_results,
        pd.DataFrame(
            conformal_results
        )
    ],
    ignore_index=True,
    sort=False
)

FINAL_CSV = (
    "Electricity_Darts_All_Baselines.csv"
)

all_baseline_results.to_csv(
    FINAL_CSV,
    index=False
)

print(
    "Saved:",
    FINAL_CSV
)

all_baseline_results


In [ ]:
# ============================================================
# CELL 28 — RANK BY CRPS
# ============================================================

ranking = (
    all_baseline_results
    .sort_values(
        "CRPS",
        ascending=True
    )
    .reset_index(drop=True)
)

columns = [
    "Model",
    "Method",
    "MAE",
    "MASE",
    "RMSSE",
    "CRPS",
    "PICP",
    "MIS",
    "MPIW",
    "Rho_Risk",
    "Training Time (s)",
    "Inference Time (s)",
]

ranking[
    [
        c for c in columns
        if c in ranking.columns
    ]
]


In [ ]:
# ============================================================
# CELL 29 — COVERAGE ANALYSIS
# ============================================================

coverage = ranking[
    [
        "Model",
        "Method",
        "PICP",
        "MPIW",
        "MIS",
        "CRPS",
    ]
].copy()

coverage[
    "Coverage Error"
] = np.abs(
    coverage["PICP"] - 0.95
)

coverage = coverage.sort_values(
    "Coverage Error"
)

coverage# ============================================================
# CELL 30 — FINAL CLEAN TABLE
# ============================================================

final_table = ranking[
    [
        "Model",
        "Method",
        "MAE",
        "MASE",
        "RMSSE",
        "CRPS",
        "PICP",
        "MIS",
        "MPIW",
        "Rho_Risk",
    ]
].copy()

final_table


In [ ]:
# ============================================================
# CELL 31 — SAVE FINAL OUTPUTS
# ============================================================

all_baseline_results.to_csv(
    "/kaggle/working/Electricity_Darts_All_Baselines.csv",
    index=False
)

ranking.to_csv(
    "/kaggle/working/Solar_Darts_Ranked.csv",
    index=False
)

coverage.to_csv(
    "/kaggle/working/Solar_Darts_Coverage.csv",
    index=False
)

print("Files saved:")
print(
    "/kaggle/working/Electricity_Darts_All_Baselines.csv"
)
print(
    "/kaggle/working/Solar_Darts_Ranked.csv"
)
print(
    "/kaggle/working/Solar_Darts_Coverage.csv"
)
